# EMA H2: desperate-direction projections, Qwen3-4B

Runs the H2 pipeline in [`ema.h2`](../src/ema/h2/). Plan: [`prereg/h2_activations.md`](../prereg/h2_activations.md).

- **`MODE = "pilot"`** (development): the model writes the direction stories, the directions are built, and the **pilot** logs in `results/logs` are replayed. Only condition-pooled diagnostics are printed. Commit `out/` to `results/h2/pilot/`; its stories and directions are then frozen.
- **`MODE = "confirmatory"`**: uses the frozen directions from the repo (no regeneration) and replays the confirmatory `oneoff` logs from the attached dataset, then runs the pre-registered analysis. Run only after the addendum is frozen and the Qwen run is complete.

**Kaggle setup:** Accelerator *GPU T4 x2*, Internet *on*. About 1 GPU-hour for the pilot mode, 2 to 3 for the confirmatory mode.

In [ ]:
import glob, os, shutil, subprocess, sys
MODE = "pilot"            # or "confirmatory"
REF = "main"              # the confirmatory mode must use the tag named in the addendum
REPO = "https://github.com/NischalParulekar/remembered-failure"
WORK = "/kaggle/working"
MODEL = "Qwen/Qwen3-4B-Instruct-2507"
OUT = f"{WORK}/out"
PY = sys.executable
os.makedirs(OUT, exist_ok=True)
os.chdir(WORK)

## 1. Environment

In [ ]:
%%bash -s "$REPO" "$WORK" "$PY" "$REF"
set -e
cd "$2"
[ -d EMA ] || git clone -q "$1" EMA
git -C EMA fetch -q --tags && git -C EMA checkout -q "$4"
pip install -q uv
uv pip install -q --python "$3" vllm -e "./EMA[dev]"
git -C EMA rev-parse HEAD | tee commit.txt
uv pip freeze --python "$3" > env.txt
grep -iE "^(vllm|inspect-ai|torch|transformers)==" env.txt

## 2. Directions
Pilot mode generates the stories (vLLM, its own process so the GPU is freed) and builds the directions. Confirmatory mode copies the frozen ones.

In [ ]:
def run(*args):
    r = subprocess.run([PY, "-X", "utf8", *args], cwd=f"{WORK}/EMA", capture_output=True, text=True,
                       env={**os.environ, "PYTHONIOENCODING": "utf-8"})
    print(r.stdout[-6000:]); print(r.stderr[-3000:])
    r.check_returncode()

if MODE == "pilot":
    run("-m", "ema.h2.run", "generate", f"{OUT}/stories.jsonl", MODEL)
    run("-m", "ema.h2.run", "directions", f"{OUT}/stories.jsonl", MODEL, OUT)
else:
    for f in ("stories.jsonl", "directions.npz", "validity.json"):
        shutil.copy2(f"{WORK}/EMA/results/h2/pilot/{f}", OUT)
    print(open(f"{OUT}/validity.json").read())

## 3. Replay

In [ ]:
if MODE == "pilot":
    for split in ("oneoff", "conflicting"):
        run("-m", "ema.h2.run", "replay", f"{WORK}/EMA/results/logs", split, MODEL, OUT)
else:
    found = glob.glob("/kaggle/input/**/eval-set.json", recursive=True)
    assert found, "attach the confirmatory logs dataset"
    run("-m", "ema.h2.run", "replay", os.path.dirname(found[0]), "oneoff", MODEL, OUT)

## 4. Results
Pilot: condition-pooled turn profiles only (no condition contrast). Confirmatory: the pre-registered analysis.

In [ ]:
if MODE == "pilot":
    for split in ("oneoff", "conflicting"):
        print("==", split); run("-m", "ema.h2.stats_h2", "--pooled", f"{OUT}/projections_{split}.jsonl")
else:
    run("-m", "ema.h2.stats_h2", f"{OUT}/projections_oneoff.jsonl")
shutil.copy2(f"{WORK}/commit.txt", OUT); shutil.copy2(f"{WORK}/env.txt", OUT)